# Error Analysis


## Setup & Imports

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"          # global (frozen prompts)

# UNI-88: point at one experiment folder. Paste the name printed by pipeline.ipynb §2.
EXPERIMENT_NAME = "experiment_test_9385_20260609_1013"       # <-- set to the run you are analysing
EXPERIMENT_DIR  = ROOT / "data" / "experiments" / EXPERIMENT_NAME
LLAMA_RUNS  = EXPERIMENT_DIR / "llama_runs"
EXPERIMENT  = EXPERIMENT_DIR / "experiment.jsonl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
print(f"ROOT = {ROOT}")

ROOT = /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis


### Analysis 1 — Structural confusion: do wrong retrievals share a plot skeleton?

**Goal:** When the structural model retrieves the wrong story, is it because of a random error/noise, or whether the errors are meaningful where these "wrong" stories are structurally similar (e.g., share same archetypical event trigger and/or event type skeleton)? 

**Method:**
1. Take every query whose top-1 retrieval belongs to a different work, so the actual errors hapenned. Call the wrongly retrieved summary "wrong match". 
2. For each (query, wrong match) pair, measure how much narrative structure the two share, in four ways: 
    1. Shared *event types* via Traversky Index overlap: How much *event types* overlap between query and the wrong match?
    2. Longest in-order common event type sequence (namely, LCS ratio) for *event types*: What is the longest sequence of *event types* that appear in both the query and the wrong match? 
    3. Shared *event triggers* via Traversky Index overlap: How much *event triggers* overlap between query and the wrong match?
    4. Longest in-order common event type sequence (namely, LCS ratio) for *event triggers*: What is the longest sequence of *event triggers* that appear in both the query and the wrong match?
3. Build a chance baseline: pair the same queries with a random story (also a different work) and compute the same four measures. These are the ratios created, as if we would do things randomly.
4. Compare error pairs vs random pairs (one-sided Mann–Whitney U): if the scores of match between (query, wrong match) is higher than the scores of match (query, random match), the errors are systematic structural confusion (i.e., signal)
5. Visually inspect the top pairs by hand: the shared event sequence (the candidate plot skeleton), genres, languages, and the two texts side by side.

**Results:**
- 77.5% of queries (4,393 / 5,666) retrieve a wrong story at rank 1 (Qwen3 × events_only).
- The wrong story shares **almost twice** the event categories with the query that a random story would (Dice 0.44 vs 0.23).
- The same holds for **order**: the shared in-order event sequence is ~2× longer than chance (0.20 vs 0.11) — so it's plot *shape*, not just shared ingredients.
- Even on **exact trigger verbs**, the wrong match is ~3× above chance (0.10 vs 0.04) — so the effect is not an artifact of coarse or noisy event-type labels.
- All four differences are significant at p ≈ 0. Sanity check: the random baseline for event-type overlap (0.23) reproduces the corpus-wide mean from pipeline §8.8 / Table 5 (0.237).
- Top confusions share interpretable skeletons across languages and genres — e.g. *Sending → Arriving → Giving → Warning* shared by a German doctor drama and an Italian mafia film with **zero** shared genres.

In [2]:
import random

import numpy as np
from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu

# Set up encoder and condition. Although all conditions have the same set of events, the retrieval results, therefore the wrong match, changes across conditions. 
#   VIEW (embedder): qwen3_emb_0p6b | e5_mistral
#   COND (representation): events_only | temporal | causal | temporal_causal_independent | temporal_causal_joint | raw_text
VIEW, COND = "qwen3_emb_0p6b", "events_only"
SEED = 0
# Download the data from cache, instead of re-loading the full data in each run to save time
META_CACHE   = EXPERIMENT_DIR / "error_analysis_meta.jsonl"
# Get the similarities across all conditions and embedders
SIMILARITIES = EXPERIMENT_DIR / "similarities.npz"

# 1. Get per-row metadata only, since experiment.jsonl carries embeddings (~4.4 GB)
if not META_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(META_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            out.write(json.dumps({
                "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
                "lang": r.get("lang"), "genres": r.get("genres") or [],
                "triggers": [e["trigger"].lower() for e in r.get("events") or []],
                "types":    [e["event_type"]      for e in r.get("events") or []],
                "text": r.get("text", ""),
            }, ensure_ascii=False) + "\n")
meta  = [json.loads(l) for l in META_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
works = np.array([m["wikidata_id"] for m in meta])
N     = len(meta)

# 2. Rank-1 neighbor per query (similarities.npz rows follow experiment.jsonl order; diagonal is -inf)
sim = np.load(SIMILARITIES)[f"{VIEW}__{COND}"]
assert sim.shape == (N, N), f"similarities {sim.shape} vs meta rows {N} — cache stale? delete {META_CACHE}"
rank1 = sim.argmax(axis=1)
wrong = np.flatnonzero(works[rank1] != works)
error_pairs = [(int(q), int(rank1[q])) for q in wrong]
print(f"{VIEW}__{COND}: wrong story at rank 1 for {len(wrong)}/{N} queries ({len(wrong)/N:.1%})")

# 3. Skeleton-overlap measures. Inventory = set-based Dice; order = LCS ratio with the same normalization (2·LCS/(|a|+|b|)), so inventory and order numbers are directly comparable.
def dice(a, b):
    A, B = set(a), set(b)
    return 2 * len(A & B) / (len(A) + len(B)) if (A or B) else 0.0

def lcs_table(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            L[i + 1][j + 1] = L[i][j] + 1 if x == y else max(L[i][j + 1], L[i + 1][j])
    return L

def lcs_ratio(a, b):
    return 2 * lcs_table(a, b)[-1][-1] / (len(a) + len(b)) if (a or b) else 0.0

# Calculate the measures
MEASURES = {
    "Event-type inventory overlap": lambda u, v: dice(u["types"], v["types"]),
    "Event-type order overlap":     lambda u, v: lcs_ratio(u["types"], v["types"]),
    "Trigger inventory overlap":    lambda u, v: dice(u["triggers"], v["triggers"]),
    "Trigger order overlap":        lambda u, v: lcs_ratio(u["triggers"], v["triggers"]),
}

# 4. Random (Null) Baseline: same queries, but each paired with a random non-relevant candidate instead of the model's wrong choice, basically this is what overlap looks like when the pairing carries no signal
rng = random.Random(SEED)
null_pairs = []
for q, _ in error_pairs:
    j = rng.randrange(N)
    while j == q or works[j] == works[q]:
        j = rng.randrange(N)
    null_pairs.append((q, j))

# 5. Error vs null per measure
scores, table = {}, []
for name, fn in MEASURES.items():
    err = np.array([fn(meta[q], meta[c]) for q, c in error_pairs])
    nul = np.array([fn(meta[q], meta[c]) for q, c in null_pairs])
    _, p = mannwhitneyu(err, nul, alternative="greater")
    scores[name] = (err, nul)
    table.append({"Measure": name,
                  "Error pair mean":  f"{err.mean():.4f}",
                  "Random-pair mean": f"{nul.mean():.4f}",
                  "Δ":                f"{err.mean() - nul.mean():+.4f}",
                  "p":                "< .001" if p < .001 else f"{p:.3f}"})
    
# Display the table, and a note for my thesis and other readers.
display(pd.DataFrame(table))
display(Markdown(
    f"*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under "
    f"*{VIEW} × {COND}* (embedder × representation condition; set `VIEW`/`COND` above to analyse "
    f"another cell of the ablation grid). A *random pair* is the same query paired with a randomly "
    f"drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient "
    f"(symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. "
    f"*Order overlap* respects narrative order: the longest common subsequence between the two item "
    f"sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. "
    f"Both are computed over MAVEN event types and over exact (lowercased) trigger words. "
    f"*p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs."
))

qwen3_emb_0p6b__events_only: wrong story at rank 1 for 4393/5666 queries (77.5%)


,Measure,Error pair mean,Random-pair mean,Δ,p
0,Event-type inventory overlap,0.4363,0.2282,+0.2081,< .001
1,Event-type order overlap,0.2031,0.1105,+0.0925,< .001
2,Trigger inventory overlap,0.1008,0.0359,+0.0649,< .001
3,Trigger order overlap,0.0657,0.0256,+0.0401,< .001


*Note.* An *error pair* is a query and the wrong story retrieved for it at rank 1 under *qwen3_emb_0p6b × events_only* (embedder × representation condition; set `VIEW`/`COND` above to analyse another cell of the ablation grid). A *random pair* is the same query paired with a randomly drawn summary of a different work. *Inventory overlap* ignores order: the Dice coefficient (symmetric Tversky index, α=β=0.5) between the two summaries' sets of items. *Order overlap* respects narrative order: the longest common subsequence between the two item sequences, normalized as *2·LCS/(|a|+|b|)*, so it is directly comparable to inventory overlap. Both are computed over MAVEN event types and over exact (lowercased) trigger words. *p*: one-sided Mann–Whitney U, H1 = error pairs overlap more than random pairs.

### 1.2 Stratified robustness
**Goal:** is the structural-confusion effect general, or an artifact of short summaries? Queries are split into quartiles by their number of extracted events, and the same error-vs-random comparison is run within each quartile. Δ > 0 with a significant p in EVERY quartile (incl. the longest summaries) ⇒ the effect is general.

In [3]:
q_events = np.array([len(meta[q]["types"]) for q, _ in error_pairs])   # event count of each error query
edges    = np.quantile(q_events, [0, .25, .5, .75, 1.0])
qbin     = np.clip(np.searchsorted(edges, q_events, side="right") - 1, 0, 3)   # quartile index 0..3

rows = []
for b in range(4):
    idx = np.flatnonzero(qbin == b)
    lo, hi = int(q_events[idx].min()), int(q_events[idx].max())
    for name in MEASURES:
        err, nul = scores[name]            # full arrays, aligned with error_pairs
        e, n = err[idx], nul[idx]
        _, p = mannwhitneyu(e, n, alternative="greater")
        rows.append({"Quartile": f"Q{b+1} ({lo}–{hi} events)", "n": len(idx), "Measure": name,
                     "Error pair mean":  f"{e.mean():.4f}",
                     "Random-pair mean": f"{n.mean():.4f}",
                     "Δ":                f"{e.mean() - n.mean():+.4f}",
                     "p":                "< .001" if p < .001 else f"{p:.3f}"})
display(pd.DataFrame(rows).set_index(["Quartile", "n", "Measure"]))
display(Markdown(
    f"*Note.* Same error and random pairs as the table above ({VIEW} × {COND}), stratified into "
    f"quartiles by the **query's number of extracted events** (a length proxy that is also the unit "
    f"the overlap measures operate on). Because each random pair reuses its error pair's query, the "
    f"two groups are identical on the query side within every quartile. A positive Δ with significant "
    f"*p* in all four quartiles shows the structural-confusion effect is general across summary "
    f"lengths, not an artifact of short summaries (overlap *ratios* are mechanically larger for "
    f"short event sequences, but that inflation applies to error and random pairs alike)."
))

Error pair mean Random-pair mean        Δ       p
Quartile           n    Measure                                                                       
Q1 (5–10 events)   1032 Event-type inventory overlap          0.3625           0.1201  +0.2425  < .001
                        Event-type order overlap              0.2287           0.0757  +0.1530  < .001
                        Trigger inventory overlap             0.0849           0.0166  +0.0683  < .001
                        Trigger order overlap                 0.0736           0.0148  +0.0588  < .001
Q2 (11–24 events)  1127 Event-type inventory overlap          0.3799           0.1962  +0.1837  < .001
                        Event-type order overlap              0.1885           0.1076  +0.0810  < .001
                        Trigger inventory overlap             0.0864           0.0289  +0.0575  < .001
                        Trigger order overlap                 0.0624           0.0233  +0.0391  < .001
Q3 (25–51 events)  1108 Event-type inventory overlap          0.4601           0.2762  +0.1840  < .001
                        Event-type order overlap              0.1929           0.1289  +0.0640  < .001
                        Trigger inventory overlap             0.1023           0.0428  +0.0596  < .001
                        Trigger order overlap                 0.0619           0.0299  +0.0320  < .001
Q4 (52–126 events) 1126 Event-type inventory overlap          0.5368           0.3121  +0.2246  < .001
                        Event-type order overlap              0.2041           0.1273  +0.0768  < .001
                        Trigger inventory overlap             0.1281           0.0539  +0.0742  < .001
                        Trigger order overlap                 0.0655           0.0334  +0.0321  < .001

*Note.* Same error and random pairs as the table above (qwen3_emb_0p6b × events_only), stratified into quartiles by the **query's number of extracted events** (a length proxy that is also the unit the overlap measures operate on). Because each random pair reuses its error pair's query, the two groups are identical on the query side within every quartile. A positive Δ with significant *p* in all four quartiles shows the structural-confusion effect is general across summary lengths, not an artifact of short summaries (overlap *ratios* are mechanically larger for short event sequences, but that inflation applies to error and random pairs alike).

### 1.3 Visually check the cases

In [4]:
from IPython.display import display, Markdown

TOP_K   = 3
PREVIEW = None   # chars of raw text shown per summary; None = full text (for the thesis appendix)

def cut(s):
    return s if PREVIEW is None or len(s) <= PREVIEW else s[:PREVIEW] + "…"

def lcs_seq(a, b):
    """Backtrack the DP table to recover one LCS (the shared skeleton itself)."""
    L = lcs_table(a, b)
    i, j, out = len(a), len(b), []
    while i and j:
        if a[i-1] == b[j-1]:
            out.append(a[i-1]); i -= 1; j -= 1
        elif L[i-1][j] >= L[i][j-1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]

err_lcs = scores["Event-type order overlap"][0]
order   = np.argsort(-err_lcs)

md = [f"## Top {TOP_K} structural confusions under `{VIEW}__{COND}` (by event-type order overlap)\n"]
for rank, r in enumerate(order[:TOP_K], 1):
    q, c = error_pairs[r]
    mq, mc = meta[q], meta[c]
    shared_skel   = lcs_seq(mq["types"], mc["types"])
    shared_genres = sorted(set(mq["genres"]) & set(mc["genres"]))
    md.append(
        f"### {rank}. `{mq['wikidata_id']}__{mq['summary_id']}` → `{mc['wikidata_id']}__{mc['summary_id']}`"
        f" &nbsp; (cosine similarity {sim[q, c]:.3f}, event type order overlap {err_lcs[r]:.3f}, "
        f"event type inventory {dice(mq['types'], mc['types']):.3f}, event trigger inventory overlap {dice(mq['triggers'], mc['triggers']):.3f})\n\n"
        f"- **shared event-type skeleton ({len(shared_skel)} types)**: {' → '.join(shared_skel)}\n"
        f"- **shared genres**: {', '.join(shared_genres) if shared_genres else '(none)'}"
        f" &nbsp;|&nbsp; langs: {mq['lang']} vs {mc['lang']}\n\n"
        f"- **Event Types of the Query ({len(mq['types'])})**: {' → '.join(mq['types'])}\n"
        f"- **Event Types of the Wrong Match ({len(mc['types'])})**: {' → '.join(mc['types'])}\n"
        f"- **Event Triggers of the Query:** {', '.join(mq['triggers'])}\n"
        f"- **Event Triggers of the Wrong Match**: {', '.join(mc['triggers'])}\n"
        f"- **Text of the Query**: {cut(mq['text'])}\n"
        f"- **Text of the Wrong Match**: {cut(mc['text'])}\n"
    )
display(Markdown("\n".join(md)))

## Top 3 structural confusions under `qwen3_emb_0p6b__events_only` (by event-type order overlap)

### 1. `5504130__fr` → `562586__fr` &nbsp; (cosine similarity 0.797, event type order overlap 0.727, event type inventory 0.727, event trigger inventory overlap 0.182)

- **shared event-type skeleton (4 types)**: Come_together → Departing → Becoming → Know
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs fr

- **Event Types of the Query (5)**: Come_together → Departing → Becoming → Giving → Know
- **Event Types of the Wrong Match (6)**: Destroying → Come_together → Departing → Becoming → Know → GetReady
- **Event Triggers of the Query:** meets, leave, became, gave, find
- **Event Triggers of the Wrong Match**: decimated, meets, left, turn, discover, prepares
- **Text of the Query**: A young man, Paul Harrison, the son of a wealthy British businessman living in Paris, meets a beautiful young orphan, Michelle Latour. The two teenagers leave Paris for the Camargue. Michelle became pregnant and gave birth to a baby girl. The young couple and the child lead a family life until the police find them.
- **Text of the Wrong Match**: An army veteran, the sole survivor of a decimated American patrol, meets a young South Korean, Short Round, as well as others left behind by the war. He leads them to an unoccupied Buddhist temple, which they turn into an observation camp. But when they discover that they are in the immediate vicinity of a North Korean communist camp, the troop prepares for the possibility of a fight...

### 2. `76940500__en` → `1029697__fr` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: en vs fr

- **Event Types of the Query (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Types of the Wrong Match (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Triggers of the Query:** found, sent, release, return, giving, threatened
- **Event Triggers of the Wrong Match**: transferred, arrives, gives, warning, placed, becomes
- **Text of the Query**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.
- **Text of the Wrong Match**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.

### 3. `1029697__fr` → `76940500__en` &nbsp; (cosine similarity 0.811, event type order overlap 0.667, event type inventory 0.667, event trigger inventory overlap 0.000)

- **shared event-type skeleton (4 types)**: Sending → Arriving → Giving → Warning
- **shared genres**: (none) &nbsp;|&nbsp; langs: fr vs en

- **Event Types of the Query (6)**: Sending → Arriving → Giving → Warning → Placing → Becoming
- **Event Types of the Wrong Match (6)**: Know → Sending → Releasing → Arriving → Giving → Warning
- **Event Triggers of the Query:** transferred, arrives, gives, warning, placed, becomes
- **Event Triggers of the Wrong Match**: found, sent, release, return, giving, threatened
- **Text of the Query**: Commissioner Betti is being transferred to Naples on orders from his hierarchy. As soon as he arrives on the scene, the local mafia gives him a threatening welcome in the form of a warning. The latter is placed under the  di 'O Generale  cup. At his instigation, the organization becomes the scene of a fratricidal reckoning.
- **Text of the Wrong Match**: Doctor Gerda Maurer is found guilty of negligence and sent to prison. On her release she cannot return to her career in medicine and has to take on other jobs such as a nightclub singer. She is giving a second chance by a respected surgeon and works as his assistant, but her new position is threatened by the return of her former lover Georg.


### Analysis 2 — Structural mismatch: why does adding relations make retrieval worse?

**Goal:** Adding temporal/causal relations consistently *lowers* retrieval performance relative to events-only, so we find the mechanism: does the relation layer add discriminative signal, or does it just make all stories look more alike?

**What the relation layer actually adds.** Relations are linearized in the hybrid format, e.g. *(e2:left|Departing, BEFORE, e1:arrived|Arriving)* — endpoints carry trigger and type. So, relative to the events-only string, the relation block adds three things:
- (a) **Relation labels** (e.g., *BEFORE*, *CAUSE*, ...), drawn from a taxonomies from literature
- (b) **Repeated copies of event tokens** the *EVENTS:* block already contains: an event linked in 10 relations gets its trigger/type repeated 10 times, re-weighting events by how often Llama links them, not by narrative importance
- (c) **Actual relation pattern** (which event pairs connect) — the only genuinely new information, and only useful if it is not predictable from event order

**Method:**
1. **Collapse check:** Mean and SD of the pairwise cosine similarity across all summaries, per condition. If the mean rises and the SD shrinks as relations are added, the relation layer pushes all stories closer together and compresses the spread that retrieval depends on.
2. **Label diversity:** — The distribution of relation labels per condition. If one label (e.g. *BEFORE*) dominates, the labels cannot discriminate between stories.
3. **Link-pattern predictability:** — The share of relations connecting adjacent events (*e_i → e_(i+1)*). If most links are adjacent-pair chains, the link pattern is derivable from event order and adds nothing new.
4. **Casualties:** — Queries where events-only retrieved the right story at rank 1 but a relation condition put a wrong story on top; inspect whether the wrong match wins on relation-pattern similarity despite different events.

**Results**


### 2.1 Collapse check
**Goal:** Show whether each added relation layer pushes *all* summaries closer together in embedding space (mean pairwise similarity up, spread down), instead of separating related from unrelated ones.

In [5]:
# Mean / SD of pairwise cosine similarity per encoder × condition, straight from similarities.npz.
# Rising mean + shrinking SD as relations are added = the space collapses (sameness, not signal).
sims_npz = np.load(SIMILARITIES)
rows = []
for view in ["qwen3_emb_0p6b", "e5_mistral"]:
    for cond in ["events_only", "temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]:
        m   = sims_npz[f"{view}__{cond}"]
        off = m[np.isfinite(m)]            # diagonal is -inf (self-similarity masked)
        rows.append({"Encoder": view, "Condition": cond,
                     "Mean pairwise cosine": f"{off.mean():.4f}",
                     "SD": f"{off.std():.4f}"})
display(pd.DataFrame(rows).set_index(["Encoder", "Condition"]))

Mean pairwise cosine      SD
Encoder        Condition                                               
qwen3_emb_0p6b events_only                               0.6179  0.0946
               temporal                                  0.6447  0.0899
               causal                                    0.6491  0.0891
               temporal_causal_independent               0.6579  0.0847
               temporal_causal_joint                     0.6685  0.0801
e5_mistral     events_only                               0.8876  0.0282
               temporal                                  0.9057  0.0384
               causal                                    0.9135  0.0280
               temporal_causal_independent               0.9220  0.0425
               temporal_causal_joint                     0.9157  0.0324

### 2.2 Relation-label diversity
**Goal:** Check whether the relation labels are diverse enough to distinguish stories, or whether one generic label (e.g. *BEFORE*) dominates everything.

In [6]:
# Relation-label distribution per condition. Relations live in experiment.jsonl (4.4 GB),
# so they are streamed once into their own light cache, like the metadata in Analysis 1.
from collections import Counter

REL_CACHE   = EXPERIMENT_DIR / "error_analysis_relations.jsonl"
LLAMA_CONDS = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]

if not REL_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(REL_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            row = {"wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"], "conditions": {}}
            for cond in LLAMA_CONDS:
                rel = (r.get("conditions", {}).get(cond) or {}).get("relations") or {}
                row["conditions"][cond] = [[t["source"], t["relation"], t["target"]]
                                           for trs in rel.values() for t in (trs or [])]
            out.write(json.dumps(row) + "\n")
rels = [json.loads(l) for l in REL_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(rels) == N, f"relations cache has {len(rels)} rows vs meta {N} — delete {REL_CACHE} and re-run"

rows = []
for cond in LLAMA_CONDS:
    cnt   = Counter(lab for row in rels for _, lab, _ in row["conditions"][cond])
    total = sum(cnt.values())
    for lab, c in cnt.most_common():
        rows.append({"Condition": cond, "Label": lab, "Count": c, "Share": f"{c/total:.1%}"})
display(pd.DataFrame(rows).set_index(["Condition", "Label"]))

Count  Share
Condition                   Label                               
temporal                    BEFORE                 122923  84.9%
                            IDENTITY                13691   9.5%
                            CONTAINS                 4661   3.2%
                            OVERLAPS                 3493   2.4%
causal                      CAUSE                   71632  91.3%
                            ENABLE                   3739   4.8%
                            PREVENT                  1609   2.1%
                            CAUSE_TO_END             1480   1.9%
temporal_causal_independent BEFORE                 122923  55.1%
                            CAUSE                   71632  32.1%
                            IDENTITY                13691   6.1%
                            CONTAINS                 4661   2.1%
                            ENABLE                   3739   1.7%
                            OVERLAPS                 3493   1.6%
                            PREVENT                  1609   0.7%
                            CAUSE_TO_END             1480   0.7%
temporal_causal_joint       CAUSE_BEFORE            78585  72.9%
                            CAUSE_TO_END_BEFORE      9921   9.2%
                            CAUSE_OVERLAPS           6074   5.6%
                            ENABLE_BEFORE            6014   5.6%
                            BEFORE                   2223   2.1%
                            PREVENT_BEFORE           2135   2.0%
                            CAUSE_TO_END_DURING      1111   1.0%
                            CONTAINS                  502   0.5%
                            ENABLE_OVERLAPS           445   0.4%
                            IDENTITY                  383   0.4%
                            CAUSE_TO_END_OVERLAPS     365   0.3%
                            PREVENT_OVERLAPS           69   0.1%
                            OVERLAPS                   27   0.0%

### 2.3 Link-pattern predictability
**Goal:** Check whether the relations mostly link adjacent events — if so, the link pattern is derivable from event order and carries almost no new information.

In [7]:
# Share of relations that just link adjacent events (e_i -> e_(i+1)), per condition.
# Uses the relations cache built in 2.2.
rows = []
for cond in LLAMA_CONDS:
    adj = fwd = total = 0
    for row in rels:
        for s, lab, t in row["conditions"][cond]:
            si, ti = int(s[1:]), int(t[1:])
            total += 1
            fwd   += si < ti
            adj   += abs(ti - si) == 1
    rows.append({"Condition": cond, "n relations": total,
                 "% adjacent (|target − source| = 1)": f"{adj/total:.1%}",
                 "% forward (source < target)":        f"{fwd/total:.1%}"})
display(pd.DataFrame(rows).set_index("Condition"))

,n relations,% adjacent (|target − source| = 1),% forward (source < target)
Condition,,,
temporal,144768,98.8%,95.7%
causal,78460,92.0%,96.2%
temporal_causal_independent,223228,96.4%,95.9%
temporal_causal_joint,107854,95.4%,94.6%


### 2.4 Casualties (right → wrong)
**Goal:** Count the queries that events-only retrieved correctly but a relation condition broke — the direct damage the relation layer causes.

In [9]:
# Per relation condition: how many queries events_only answered correctly at rank 1 become
# wrong ("broken"), how many wrong ones become right ("fixed"), and the net effect.
sims_npz   = np.load(SIMILARITIES)
BASE       = "events_only"
base_rank1 = sims_npz[f"{VIEW}__{BASE}"].argmax(axis=1)
base_right = works[base_rank1] == works

REL_CONDS = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]
rows, broken_examples = [], {}
for cond in REL_CONDS:
    r1    = sims_npz[f"{VIEW}__{cond}"].argmax(axis=1)
    broke = np.flatnonzero(base_right  & (works[r1] != works))
    fixed = np.flatnonzero(~base_right & (works[r1] == works))
    broken_examples[cond] = [(int(q), int(base_rank1[q]), int(r1[q])) for q in broke]
    rows.append({"Condition": cond,
                 "Broken (right → wrong)": len(broke),
                 "Fixed (wrong → right)":  len(fixed),
                 "Net": len(fixed) - len(broke)})
display(pd.DataFrame(rows).set_index("Condition"))
display(Markdown(
    f"*Note.* The reference condition is **{BASE}** (encoder **{VIEW}**), which answers "
    f"{int(base_right.sum())} of the {N} queries correctly at rank 1. Each relation condition is "
    f"compared against it query by query: **Broken** counts queries {BASE} answered correctly whose "
    f"rank-1 retrieval becomes a wrong work once relations are added; **Fixed** counts queries {BASE} "
    f"got wrong that the relation condition repairs; **Net** = Fixed − Broken, so a negative value "
    f"means the relation layer breaks more retrievals than it fixes. Net / {N} equals the ΔP@1 of "
    f"Table 4 exactly — this table decomposes that aggregate delta into its gross movements: the "
    f"churn (Broken + Fixed) is several times larger than the net, i.e. relations substantially "
    f"reshuffle which stories are retrieved while slightly tilting the balance toward harm."
))

# A few broken examples: the correct story events_only found vs the wrong story the relation
# condition preferred, with event-type overlap for context.
EXAMPLE_COND, N_EXAMPLES = "temporal_causal_joint", 3
md = [f"#### Broken examples under `{VIEW}__{EXAMPLE_COND}` (events_only was right)\n"]
for q, good, bad in broken_examples[EXAMPLE_COND][:N_EXAMPLES]:
    mq, mg, mb = meta[q], meta[good], meta[bad]
    md.append(
        f"- query `{mq['wikidata_id']}__{mq['summary_id']}`: events_only → correct "
        f"`{mg['wikidata_id']}__{mg['summary_id']}` (type inventory {dice(mq['types'], mg['types']):.2f}); "
        f"{EXAMPLE_COND} → wrong `{mb['wikidata_id']}__{mb['summary_id']}` "
        f"(type inventory {dice(mq['types'], mb['types']):.2f})"
    )
display(Markdown("\n".join(md)))

,Broken (right → wrong),Fixed (wrong → right),Net
Condition,,,
temporal,213,158,-55
causal,268,166,-102
temporal_causal_independent,251,200,-51
temporal_causal_joint,244,185,-59


*Note.* The reference condition is **events_only** (encoder **qwen3_emb_0p6b**), which answers 1273 of the 5666 queries correctly at rank 1. Each relation condition is compared against it query by query: **Broken** counts queries events_only answered correctly whose rank-1 retrieval becomes a wrong work once relations are added; **Fixed** counts queries events_only got wrong that the relation condition repairs; **Net** = Fixed − Broken, so a negative value means the relation layer breaks more retrievals than it fixes. Net / 5666 equals the ΔP@1 of Table 4 exactly — this table decomposes that aggregate delta into its gross movements: the churn (Broken + Fixed) is several times larger than the net, i.e. relations substantially reshuffle which stories are retrieved while slightly tilting the balance toward harm.

#### Broken examples under `qwen3_emb_0p6b__temporal_causal_joint` (events_only was right)

- query `1064885__de`: events_only → correct `1064885__it` (type inventory 0.38); temporal_causal_joint → wrong `1379881__de` (type inventory 0.41)
- query `107304169__de`: events_only → correct `107304169__en` (type inventory 0.54); temporal_causal_joint → wrong `464032__fr` (type inventory 0.49)
- query `107304169__en`: events_only → correct `107304169__de` (type inventory 0.54); temporal_causal_joint → wrong `15910472__de` (type inventory 0.42)

## (iv) Extraction Error

**Goal:** The events themselves are wrong — missing, noisy, or mislabeled — so the structural representation is broken *before* retrieval even starts.

Unlike (iii), the problem here isn't the relations — it's the **events** BERT+CRF pulled out of the text. If the events are junk, everything built on top (relations, linearized string, embedding) is junk too. Three kinds of bad event:
- **missing** — a real event the model never detected;
- **noisy** — a word tagged as an event that isn't a story event (e.g. the linking verbs `causes`, `makes`, `due`);
- **misclassified** — a real event given the wrong MAVEN type.

**What the pipeline already removed (so we don't re-count it here):**
- **Hallucinated relation IDs** — Llama inventing links to events that don't exist — were dropped in §4.6 and saved to `hallucinated_relations.jsonl`. The pattern: ~98% invent exactly `e(N+1)`, almost always linking the *last* real event — the model assumes everything must connect to something (UNI-60).
- **Broken / unparseable relations** — JSON parse failures and context-overflow summaries — were dropped whole in §4.5 (complete-case), with counts in `experiment.yaml`.

So the live work here is the **event-level noise that survived** because it looks valid.

**Method:**
1. **Count noisy triggers** — fraction of events whose trigger is a relational/linking verb (`causes`, `makes`, `enables`, `due`) or a subword fragment (`cing`); these are extraction artifacts, not story events (UNI-68; subword slicing was fixed in UNI-105 — verify residual ≈ 0).
2. **Spot misclassification** — sample events and check trigger vs assigned MAVEN type for obvious mismatches.
3. **Report the noise budget** — pull the hallucination rate from `hallucinated_relations.jsonl` and the parse/overflow drop counts from `experiment.yaml`, so structural noise is quantified end-to-end.
4. **Tie to retrieval** — on the worst-degrading queries (largest rank drop under structure vs baseline, UNI-103), inspect the events: is the failure explained by junk triggers rather than by the method itself?